In [ ]:
print(1,2,3)

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

files = {
    "ingest.py": f"{PREFIX}/01-agentic-rag/code/ingest.py",
    "rag_helper.py": f"{PREFIX}/01-agentic-rag/code/rag_helper.py",
    "evaluation_utils.py": f"{PREFIX}/04-evaluation/code/evaluation_utils.py",
}

for filename, url in files.items():
    urlretrieve(url, filename)
    print(f"Heruntergeladen: {filename}")

Heruntergeladen: ingest.py
Heruntergeladen: rag_helper.py
Heruntergeladen: evaluation_utils.py


In [ ]:
from ingest import load_faq_data
from evaluation_utils import llm_structured, calc_price

In [ ]:
from ingest import load_faq_data
documents = load_faq_data()

In [ ]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

118

In [ ]:
documents = documents_llm

In [ ]:
doc = documents[0]
doc["doc_id"] = doc.pop("id")
print(doc["question"])
print(doc["answer"])

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [ ]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [ ]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
import json

user_prompt = json.dumps(doc)

In [ ]:
user_prompt

'{"course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions.", "doc_id": "74eb249bbf"}'

In [ ]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [ ]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [ ]:
response.output_parsed.questions


['I just found this course, is it still okay to join now?',
 'Can new students still start the course late, or is it too late already?',
 'If I join after the course has started, can I still get a certificate?',
 'What do I need to do in order to be eligible for the course certificate?',
 'Is the project submission deadline the only thing I need to watch if I want a certificate?']

In [ ]:
doc

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

In [ ]:
len(documents)

118

In [ ]:
from evaluation_utils import llm_structured

In [ ]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course—can I still sign up and follow along, or is it too late to join?', 'If I join the course late, will I still be able to get a certificate somehow?', 'What do I need to do to earn the certificate if I’m joining after the course already started?', 'Is it okay to start the course now, and does that affect my chance of getting certified?', 'Does the course still accept new students, and when does the certificate deadline matter?']


In [ ]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=112, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=319)

In [ ]:
from evaluation_utils import calc_price

In [ ]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525, 'output_cost': 0.000504, 'total_cost': 0.00065925}

In [ ]:
records = []

for q in result.questions:
    records = []

    for q in result.questions:
        records.append({
            "question": q,
            "document": doc.get("doc_id", doc.get("id"))
        })

    records

records

[{'question': 'I just found this course—can I still sign up and follow along, or is it too late to join?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course late, will I still be able to get a certificate somehow?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to earn the certificate if I’m joining after the course already started?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to start the course now, and does that affect my chance of getting certified?',
  'document': '74eb249bbf'},
 {'question': 'Does the course still accept new students, and when does the certificate deadline matter?',
  'document': '74eb249bbf'}]

In [ ]:
import pandas as pd

In [ ]:
pd.DataFrame(records)

,question,document
0,I just found this course—can I still sign up a...,74eb249bbf
1,"If I join the course late, will I still be abl...",74eb249bbf
2,What do I need to do to earn the certificate i...,74eb249bbf
3,"Is it okay to start the course now, and does t...",74eb249bbf
4,"Does the course still accept new students, and...",74eb249bbf


### Generating Ground Truth for All Documents

In [ ]:
from evaluation_utils import llm_structured_retry

In [ ]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [ ]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []
    doc_id = doc.get("doc_id", doc.get("id"))

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc_id
        })

    return results, usage

In [ ]:
generate_ground_truth(doc)

([{'question': 'I just found this course—am I still allowed to join now?',
   'document': '74eb249bbf'},
  {'question': 'If I join late, is there still a way to get the certificate?',
   'document': '74eb249bbf'},
  {'question': 'Do I need to finish and submit the project before submissions close to receive the certificate?',
   'document': '74eb249bbf'},
  {'question': 'Can someone who discovers the course after it starts still participate?',
   'document': '74eb249bbf'},
  {'question': 'What’s the deadline condition for earning the certificate if I’m joining now?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=89, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=296))

In [ ]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    def generate_ground_truth(doc):
        user_prompt = json.dumps(doc)

        out, usage = llm_structured_retry(
            openai_client,
            data_gen_instructions,
            user_prompt,
            Questions
        )

        results = []
        doc_id = doc.get("doc_id", doc.get("id"))

        for q in out.questions:
            results.append({
                "question": q,
                "document": doc_id
            })

        return results, usage
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

Parallel processing

Running the calls one after another wastes most of the time waiting on the network. Each request just sits there until OpenAI responds, so we can fire several at once and wait on them together. We process the documents in parallel and track progress while the requests run.

One caution: don't open too many connections at once, or you'll hit the provider's rate limits. Five or six workers is a safe default here.

Import ThreadPoolExecutor:

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/118 [00:00<?, ?it/s]

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

590

In [ ]:
ground_truth[10]

{'question': 'How do students join the office hours or live workshop sessions if the Zoom link isn’t shared with us?',
 'document': '489dd1c9d9'}

Calculate the total cost:



In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.091668

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.091668

In [ ]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

In [ ]:
len(df_ground_truth)

590